In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, \
    matthews_corrcoef, recall_score, confusion_matrix

from torch.utils.tensorboard import SummaryWriter

import pandas as pd

import numpy as np

import os

from pathlib import Path

import seaborn as sns

from datetime import datetime as dt

import joblib

import nltk
from nltk.corpus import stopwords

In [ ]:
# Necessary resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [6]:
script_dir = Path().resolve()

# Imposta questa directory come directory corrente
os.chdir(script_dir)

In [ ]:
SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5

results_folder = os.path.join(script_dir, 'results')

models_folder = os.path.join(script_dir, r'results\Baseline\models')
os.makedirs(models_folder, exist_ok=True)

scaler = StandardScaler()

stop_words = stopwords.words('dutch')

vectorizer = TfidfVectorizer(
    stop_words = stopwords.words('dutch'),
    max_df = 0.95,
    min_df = 0.05
)


In [9]:
df = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')

labels = df.iloc[:, -1]
df = df.iloc[:, 1:-1]

In [10]:
indices = np.arange(df.shape[0])

trainval_indices, test_indices = train_test_split(
    indices,
    test_size = TEST_SIZE, 
    stratify = labels, 
    random_state = SEED
)

df_test = df.iloc[test_indices]
y_test = labels.iloc[test_indices]

df_trainval = df.iloc[trainval_indices]
y_trainval = labels.iloc[trainval_indices]


In [11]:
# Stratified KFold
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Inizializzazione
scaler = StandardScaler()
vectorizer = TfidfVectorizer()

for split, (train_indices, val_indices) in enumerate(skf.split(df_trainval, y_trainval)):
    model = DummyClassifier(strategy='stratified', random_state=SEED)
    
    df_train = df_trainval.iloc[train_indices]
    y_train = y_trainval.iloc[train_indices]
    
    X_texts = vectorizer.fit_transform(df_train['consult']).toarray()
    X_data = df_train.drop(columns='consult').values
    
    X_train = np.concatenate((X_texts, X_data), axis=1)
    
    X_train = scaler.fit_transform(X_train)
    
    # Addestra il modello
    model.fit(X_train, y_train)
    
    joblib.dump({
            'model': model,
            'scaler': scaler,
            'vectorizer': vectorizer
        },
        os.path.join(models_folder, f'split_{split}')
    )

In [12]:
writer_folder = os.path.join(results_folder, f'Baseline\\{dt.now().strftime("%Y_%m_%d_%H_%M")}')

skf = StratifiedKFold(
    n_splits = N_SPLITS,
    shuffle = True,
    random_state = SEED
)

for split, filename in enumerate(os.listdir(models_folder)):
    
    writer_path = os.path.join(writer_folder, f'split_{split}')
    writer = SummaryWriter(log_dir = writer_path)
    
    loaded_dict = joblib.load(os.path.join(models_folder, filename))
    
    model, scaler, vectorizer= loaded_dict['model'], loaded_dict['scaler'], \
        loaded_dict['vectorizer']
    
    df_val = df_trainval.iloc[val_indices]
    y_val = y_trainval.iloc[val_indices]
    
    X_texts = vectorizer.transform(df_val['consult']).toarray()
    X_data = df_val.drop(columns='consult').values
    
    X_val = np.concatenate((X_texts, X_data), axis=1)
    
    X_val = scaler.transform(X_val)
    
    val_pred = model.predict(X_val)
    
    acc = accuracy_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred, average='macro')  # Puoi cambiare 'macro' con 'micro', 'weighted', o 'binary' se necessario
    prc = precision_score(y_val, val_pred, average='macro')  # Analogamente, scegli l'average adatto
    mcc = matthews_corrcoef(y_val, val_pred)
    rec = recall_score(y_val, val_pred, average='macro')
    cm = confusion_matrix(y_val, val_pred)
    
    # Log to TensorBoard
    writer.add_scalar('Accuracy/val',  acc)
    writer.add_scalar('F1_Score/val',  f1)
    writer.add_scalar('Precision/val', prc)
    writer.add_scalar('MCC_Score/val', mcc)
    writer.add_scalar('Recall/val',    rec)
    writer.add_figure('Confusion_matrix/val', sns.heatmap(cm, annot=True).get_figure())
    
    X_texts = vectorizer.transform(df_test['consult']).toarray()
    X_data = df_test.drop(columns='consult').values
    
    X_test = np.concatenate((X_texts, X_data), axis=1)
    
    X_test = scaler.transform(X_test)
    
    test_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred, average='macro')  # Puoi cambiare 'macro' con 'micro', 'weighted', o 'binary' se necessario
    prc = precision_score(y_test, test_pred, average='macro')  # Analogamente, scegli l'average adatto
    mcc = matthews_corrcoef(y_test, test_pred)
    rec = recall_score(y_test, test_pred, average='macro')
    cm = confusion_matrix(y_test, test_pred)
    
    # Log to TensorBoard
    writer.add_scalar('Accuracy/test',  acc)
    writer.add_scalar('F1_Score/test',  f1)
    writer.add_scalar('Precision/test', prc)
    writer.add_scalar('MCC_Score/test', mcc)
    writer.add_scalar('Recall/test',    rec)
    writer.add_figure('Confusion_matrix/test', sns.heatmap(cm, annot=True).get_figure())